# 🧪 W12-D2 概念实验：为什么 MallSenseAI 是定时截图，不是视频流？

> 配套阅读：`第12周-Day2-仓库精读-截图vs视频流.md`（结论与 ADR 依据在那边）
> 这个 notebook 只做一件事：**用可执行的实验验证"状态型 vs 事件型"这个分水岭**
>
> 实验环境：纯 Python 模拟，不依赖真实摄像头。

## 实验 1：同一个采样策略，两种物理世界

消防通道堆货（状态型）：物理变化以小时计。
有人跌倒（事件型）：持续 5 秒，不可重现。

都用"每 2 小时拍一张"来观测，检测率天差地别。

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

def simulate_state_scenario(n_trials=10000, block_duration_h=3.0, interval_h=2.0):
    """状态型：通道被堵 block_duration_h 小时，每 interval_h 巡查一次，拍到了吗？"""
    # 事件起点随机，观测点等间隔。只要存在一个观测点落在 [start, start+duration) 内即命中
    starts = rng.uniform(0, interval_h, n_trials)   # 只需在一个间隔内考虑相位
    hits = 0
    for s in starts:
        t = 0.0
        while t < 24.0:
            if s <= t < s + block_duration_h:
                hits += 1
                break
            t += interval_h
    return hits / n_trials

def simulate_event_scenario(n_trials=10000, event_duration_s=5.0, interval_h=2.0):
    """事件型：跌倒持续 5 秒，每 2 小时拍一张，拍到的概率是多少？"""
    interval_s = interval_h * 3600
    # 观测点落在事件窗口内的概率 = event_duration / interval（事件随机出现在间隔中）
    return event_duration_s / interval_s

p_state = simulate_state_scenario()
p_event = simulate_event_scenario()
print(f"状态型（堵 3h，每 2h 巡查）   检测率 ≈ {p_state:.2%}")
print(f"事件型（跌倒 5s，每 2h 拍一张）检测率 ≈ {p_event:.6%}")
print()
print("结论：同一套采样策略，状态型场景几乎全覆盖，事件型场景等于瞎子。")

## 实验 2：采样定理直觉版 —— 采样间隔必须小于事件持续时间的 2 倍

通信里的 Nyquist 定理在这里的业务版翻译：**想靠采样抓住一个事件，采样间隔 ≤ 事件持续时间的一半**。

消防通道堆货 3 小时 → 采样间隔 < 1.5h 就可靠。跌倒 5 秒 → 采样间隔 < 2.5 秒 ≈ 已经是"连续视频"了。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

event_durations = np.array([0.01, 0.05, 0.2, 1, 3, 8, 24, 72])  # 小时：打架→跌倒→入侵→堆货→装修垃圾
required_interval = event_durations / 2

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(event_durations, required_interval, "o-", label="可靠采样所需最大间隔（=时长/2）")
ax.axhline(2.0, color="red", ls="--", label="MallSenseAI 当前：2 小时/张")
ax.axvline(24/25/3600*3600, color="gray", ls=":", alpha=0)  # 占位
for d, label in [(5/3600, "跌倒 5s"), (0.5, "入侵 30min"), (72, "装修垃圾 3天")]:
    ax.axvline(d, color="gray", ls=":", alpha=0.6)
    ax.annotate(label, (d, 0.02), rotation=90, fontsize=8, color="gray",
                xycoords="data", ha="right")
ax.set_xlabel("物理事件持续时间（小时，log）")
ax.set_ylabel("采样间隔（小时，log）")
ax.set_title("状态型场景采样绰绰有余，事件型场景必须连续观测")
ax.legend()
plt.tight_layout()
plt.show()

print("读图：红线下方 = 2小时采样可靠覆盖的区域（慢变化场景）")
print("跌倒(≈0.0014h) 远在红线上方覆盖不到的位置 → 需要视频流 + Tracking")

## 实验 3：成本账 —— 采样 vs 视频流的数据量差 5 个数量级

视频流方案不是"更好的截图"，是另一个成本宇宙。用大华 RTSP 1080P 主码流的典型参数算一笔账。

In [ ]:
# 单摄像头数据量对比（估算）
snapshot_jpeg_kb = 300          # 一张 1080P JPEG ≈ 300KB
snapshots_per_day = 12          # 每 2 小时一次 = 12 张/天

stream_mbps = 4                 # 1080P H.264 主码流 ≈ 4Mbps
hours_per_day = 24

snapshot_per_day_gb = snapshot_jpeg_kb * snapshots_per_day / 1e6
stream_per_day_gb = stream_mbps / 8 * 3600 * hours_per_day / 1000

print(f"截图方案：{snapshot_per_day_gb:.3f} GB/摄像头/天")
print(f"视频流方案：{stream_per_day_gb:.1f} GB/摄像头/天")
print(f"倍数差：{stream_per_day_gb/snapshot_per_day_gb:,.0f} 倍")
print()
print(f"100 个点位的月存储：截图 {snapshot_per_day_gb*30*100/1000:.1f} TB vs 视频流 {stream_per_day_gb*30*100/1000:.0f} TB")
print("（还没算 GPU 解码/Tracking 的算力成本——那是更大的差距）")

## 实验 4：接口即世界观 —— `capture_snapshot()` 的 ABC 只能长这样

读 `backend/app/camera/adapter.py`：`CameraAdapter` 抽象类只有一个采集方法 `capture_snapshot() -> bytes`。
这不是偷懒，是**把"截图世界观"固化进了契约**——所有 detector 下游只吃单张 JPEG。

下面验证：这个接口对状态型场景足够表达，对事件型场景天然残缺。

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass

@dataclass
class DetectionEvent:
    camera_id: str
    timestamp: float
    scene: str        # fire_corridor / floor_dirty / renovation_debris
    confidence: float

class CameraAdapter(ABC):
    """LangChat MallSenseAI 的世界观：世界是一系列快照"""
    @abstractmethod
    def capture_snapshot(self) -> bytes: ...

class MockCamera(CameraAdapter):
    def __init__(self, camera_id, world):
        self.camera_id = camera_id
        self.world = world  # world(t) -> state
    def capture_snapshot(self) -> bytes:
        # 真实实现：HTTP 抓 JPEG；mock：返回当前世界状态
        t = self.world.now
        return f"JPEG@{t}|scene={self.world.state_at(t)}".encode()

# 状态型 detector：吃单张图就够
class FireCorridorDetector:
    def detect(self, snapshot: bytes, camera_id: str) -> DetectionEvent | None:
        scene = snapshot.decode().split("scene=")[1]
        blocked = "blocked" in scene
        return DetectionEvent(camera_id, 0.0, "fire_corridor", 0.92) if blocked else None

# 尝试表达"跌倒"：需要前后帧 + 轨迹 —— 单帧接口给不了
class FallDetector:
    """跌倒检测需要连续帧（人体姿态时序变化），单张截图无法区分'躺着'和'摔倒'"""
    def detect(self, snapshot: bytes) -> None:
        raise NotImplementedError(
            "单帧无法检测跌倒：需要 frame sequence + tracking，"
            "capture_snapshot() 契约表达不了 —— 这正是接口即世界观")

world = type("W", (), {"now": 3600.0, "state_at": lambda self, t: "fire_corridor/blocked"})()
cam = MockCamera("CAM-A101-exit", world)
det = FireCorridorDetector()
ev = det.detect(cam.capture_snapshot(), cam.camera_id)
print("状态型检测：", ev)
print()
try:
    FallDetector().detect(cam.capture_snapshot())
except NotImplementedError as e:
    print("事件型检测：", e)

## 结论

| | 状态型（MallSenseAI 三场景） | 事件型（跌倒/入侵） |
|---|---|---|
| 物理时间尺度 | 分钟~天 | 秒级 |
| 采样检测率 | ~100%（实验1） | ~0%（实验1） |
| 合理手段 | 定时截图 polling | 视频流 + Tracking |
| 数据量 | MB/天/点位 | GB/天/点位（实验3） |
| 接口表达 | `capture_snapshot()` 足够 | 需要帧序列契约 |

**截图不是妥协，是匹配。**（ADR-003：采样方式是实现细节，结构化事件输出才是契约。）

→ 深入阅读：同目录 `.md` 版本第 4-5 节（ADR-004 / ADR-003 原文引用 + workers 全链路）